In [1]:
import pandas as pd
import numpy as np

application = pd.read_csv(
    "../data/processed/application_bureau_prev_inst_pos.csv"
)

cc = pd.read_csv(
    "../data/raw/credit_card_balance.csv"
)

In [2]:
cc["UTILIZATION_RATIO"] = (
    cc["AMT_BALANCE"] /
    cc["AMT_CREDIT_LIMIT_ACTUAL"]
)

cc["UTILIZATION_RATIO"] = (
    cc["UTILIZATION_RATIO"]
    .replace([np.inf, -np.inf], np.nan)
)

cc["PAYMENT_RATIO"] = (
    cc["AMT_PAYMENT_CURRENT"] /
    cc["AMT_INST_MIN_REGULARITY"]
)

cc["PAYMENT_RATIO"] = (
    cc["PAYMENT_RATIO"]
    .replace([np.inf, -np.inf], np.nan)
)

cc["LATE_PAYMENT"] = (
    cc["SK_DPD"] > 0
).astype(int)

cc["LATE_30"] = (
    cc["SK_DPD"] > 30
).astype(int)

cc["LATE_90"] = (
    cc["SK_DPD"] > 90
).astype(int)

In [3]:
cc_features = cc.groupby("SK_ID_CURR").agg(

    cc_num_cards=("SK_ID_PREV", "nunique"),

    cc_avg_balance=("AMT_BALANCE", "mean"),

    cc_max_balance=("AMT_BALANCE", "max"),

    cc_avg_credit_limit=("AMT_CREDIT_LIMIT_ACTUAL", "mean"),

    cc_avg_drawings=("AMT_DRAWINGS_CURRENT", "mean"),

    cc_total_drawings=("AMT_DRAWINGS_CURRENT", "sum"),

    cc_avg_payment=("AMT_PAYMENT_CURRENT", "mean"),

    cc_total_payment=("AMT_PAYMENT_CURRENT", "sum"),

    cc_avg_utilization=("UTILIZATION_RATIO", "mean"),

    cc_max_utilization=("UTILIZATION_RATIO", "max"),

    cc_avg_payment_ratio=("PAYMENT_RATIO", "mean"),

    cc_min_payment_ratio=("PAYMENT_RATIO", "min"),

    cc_max_payment_ratio=("PAYMENT_RATIO", "max"),

    cc_avg_dpd=("SK_DPD", "mean"),

    cc_max_dpd=("SK_DPD", "max"),

    cc_late_ratio=("LATE_PAYMENT", "mean"),

    cc_late30_ratio=("LATE_30", "mean"),

    cc_late90_ratio=("LATE_90", "mean")

).reset_index()

In [4]:
cc_features = cc_features.replace(
    [np.inf, -np.inf],
    np.nan
)

In [5]:
application_final = application.merge(
    cc_features,
    on="SK_ID_CURR",
    how="left"
)

In [6]:
application_final.to_csv(
    "../data/processed/application_final.csv",
    index=False
)

In [7]:
print(application_final.shape)

print(
    [c for c in application_final.columns
     if c.startswith("cc_")]
)

(307511, 187)
['cc_num_cards', 'cc_avg_balance', 'cc_max_balance', 'cc_avg_credit_limit', 'cc_avg_drawings', 'cc_total_drawings', 'cc_avg_payment', 'cc_total_payment', 'cc_avg_utilization', 'cc_max_utilization', 'cc_avg_payment_ratio', 'cc_min_payment_ratio', 'cc_max_payment_ratio', 'cc_avg_dpd', 'cc_max_dpd', 'cc_late_ratio', 'cc_late30_ratio', 'cc_late90_ratio']
